In [0]:

print("-----GIVEN DATAFRAME-----\n\n\n")

# First DataFrame
data1 = [
    (1, "abc", 31, "abc@gmail.com"),
    (2, "def", 23, "defyahoo.com"),
    (3, "xyz", 26, "xyz@gmail.com"),
    (4, "qwe", 34, "qwegmail.com"),
    (5, "iop", 24, "iop@gmail.com")
]

columns1 = ["id", "name", "age", "email"]

df1 = spark.createDataFrame(data1, columns1)
df1.show()

# Second DataFrame
data2 = [
    (11, "jkl", 22, "abc@gmail.com", 1000),
    (12, "vbn", 33, "vbn@yahoo.com", 3000),
    (13, "wer", 27, "wer", 2000),
    (14, "zxc", 30, "zxc.com", 2000),
    (15, "lkj", 29, "lkj@outlook.com", 2000)
]

columns2 = ["id", "name", "age", "email", "salary"]

df2 = spark.createDataFrame(data2, columns2)
df2.show()


print("-----SOLUTIONS-----\n\n\n")

##Display number of partitions in df1

# print("Number of partitions in df1: ", df1.rdd.getNumPartitions()) - using rdd in serverless in databricks is restricted.

# Using a method for dataframe

from pyspark.sql import *
from pyspark.sql.functions import *

##The below returns the number of partitions as an integer python object and not as a dataframe.

numpartitions = df1.select(spark_partition_id().alias("partitionid"))\
        .distinct()\
            .count()

print("Number of partitions in df1: ",numpartitions)

##To check number of rows inside each partition to debug data skew do below. But it shows only partitions with row and doesnt show empty partitions.
##So only the rdd method returns empty physical partitions. so if really needed, request "dedicated access mode"

partition_counts = (
    df1.select(spark_partition_id().alias("partitionid"))
       .groupBy("partitionid")
       .count()
       .orderBy("partitionid")
)
partition_counts.show()

##Create a new dataframe df3 from df1, along with a new column salary, and keep it constant 1000

df3 = df1.withColumn("salary",lit(1000))
df3.show()


# append df2 and df3, and form df4

df4 = df2.union(df3)
df4.show()

# Remove records which have invalid email from df4, emails with @ are considered to be valid.


fildf = df4.filter("email like '%@%'")
fildf.show()


# Write df4 to a target location, by partitioning on salary.

spark.sql("create database if not exists scenario10db")
spark.sql("use scenario10db")

df4.write.mode("overwrite").partitionBy("salary").saveAsTable("df4_tab")


spark.sql("select * from df4_tab").show()









